## Learning Objectives

* Understand how to prepare data for logistic regression (handling missing values, stratified splitting)
* Learn to use `statsmodels` with formula notation and `C()` for categorical variables
* Interpret logistic regression coefficients and p-values generated from `statsmodels`
* Master sklearn's implementation with manual dummy variable creation
* Understand the difference between `.predict()` and `.predict_proba()` in sklearn
* Learn about regularization in logistic regression and the `C` parameter
* Compare coefficients with and without regularization
* Compare implementation differences between statsmodels and sklearn


##  Import Libraries and Load Data


In [2]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

# For statsmodels implementation
import statsmodels.formula.api as smf

# For sklearn implementation
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, accuracy_score, precision_score, recall_score

# Set display options
pd.set_option('display.max_columns', None)
plt.style.use('seaborn-v0_8-darkgrid')

print("Libraries imported successfully!")

Libraries imported successfully!


In [3]:
# Load the Titanic dataset
titanic = sns.load_dataset('titanic')

print(f"Dataset shape: {titanic.shape}")
print(f"\nFirst few rows:")
titanic.head()



Dataset shape: (891, 15)

First few rows:


,survived,pclass,sex,age,sibsp,parch,fare,embarked,class,who,adult_male,deck,embark_town,alive,alone
0,0,3,male,22.0,1,0,7.2500,S,Third,man,True,NaN,Southampton,no,False
1,1,1,female,38.0,1,0,71.2833,C,First,woman,False,C,Cherbourg,yes,False
2,1,3,female,26.0,0,0,7.9250,S,Third,woman,False,NaN,Southampton,yes,True
3,1,1,female,35.0,1,0,53.1000,S,First,woman,False,C,Southampton,yes,False
4,0,3,male,35.0,0,0,8.0500,S,Third,man,True,NaN,Southampton,no,True


##  Data Preparation

### Imputing missing values


Like linear regression, logistic regression does not tolerate missing values

We'll focus on:

- **Target variable:** `survived` (0 = died, 1 = survived)
- **Predictors:** `age`, `fare`, `sex`, `pclass`

Let's check for missing values in these columns


In [4]:
# Check for missing values in our columns of interest
columns_of_interest = ['survived', 'age', 'fare', 'sex', 'pclass']
print("Missing values in columns of interest:")
print(titanic[columns_of_interest].isnull().sum())

Missing values in columns of interest:
survived      0
age         177
fare          0
sex           0
pclass        0
dtype: int64


There are 177 missing values in Age, which is a key predictor. Dropping all rows with missing values would result in a substantial loss of useful information. A simple and reasonable approach is group-wise median imputation: fill in missing ages using the median Age within each combination of `Sex` and `Pclass`.

In [5]:
# impute median age for each combination of sex and pclass
titanic['age'] = titanic.groupby(['sex', 'pclass'])['age'].transform(lambda x: x.fillna(x.median()))


Let's check again

In [6]:
columns_of_interest = ['survived', 'age', 'fare', 'sex', 'pclass']
print("Missing values in columns of interest:")
print(titanic[columns_of_interest].isnull().sum())

Missing values in columns of interest:
survived    0
age         0
fare        0
sex         0
pclass      0
dtype: int64


### Check Target Distribution

Let’s examine the distribution of the target variable to determine whether the dataset is balanced.

In [7]:
# Quick exploratory analysis
print("Target variable distribution:")
print(titanic['survived'].value_counts())

Target variable distribution:
survived
0    549
1    342
Name: count, dtype: int64


The dataset shows moderate class imbalance (about 62% non-survived vs 38% survived). While not extreme, this imbalance suggests we should look beyond accuracy and also consider metrics such as precision, recall, and ROC-AUC.

For such dataset, we need to do **stratified splitting** to make sure the proportion of classes is preserved in both train and test sets.

### Train-Test Splitting

In [32]:
# Split into train and test sets
train_df, test_df = train_test_split(titanic, test_size=0.2, random_state=42)

##  Implementation 1: Statsmodels with Formula API

When using `statsmodels.formula.api`, we use the **`C()` notation** to indicate categorical variables:

```python
formula = 'survived ~ age + fare + C(sex) + C(pclass)'
```

**What does `C()` do?**

- Automatically **dummy-codes** categorical variables
- Creates reference categories (first category alphabetically by default)
- For `sex`: female is reference, male gets a coefficient
- For `pclass`: First is reference, Second and Third get coefficients

This is much cleaner than manually creating dummy variables!


In [9]:
# Fit logistic regression using statsmodels
formula = 'survived ~ age + fare + C(sex) + pclass'
statsmodels_logit = smf.logit(formula=formula, data=train_df).fit()

# Display the summary
print(statsmodels_logit.summary())

Optimization terminated successfully.
         Current function value: 0.442700
         Iterations 6
                           Logit Regression Results                           
Dep. Variable:               survived   No. Observations:                  712
Model:                          Logit   Df Residuals:                      707
Method:                           MLE   Df Model:                            4
Date:                Sat, 14 Feb 2026   Pseudo R-squ.:                  0.3350
Time:                        06:34:26   Log-Likelihood:                -315.20
converged:                       True   LL-Null:                       -473.99
Covariance Type:            nonrobust   LLR p-value:                 1.749e-67
                     coef    std err          z      P>|z|      [0.025      0.975]
----------------------------------------------------------------------------------
Intercept          5.1033      0.612      8.343      0.000       3.904       6.302
C(sex)[T.male]   

### Interpreting the Coefficients

**Key points to discuss (from the printed output):**

1. **Age coefficient (negative, p=0.009):** Older passengers had lower survival odds.
2. **Fare coefficient (near 0, p=0.648):** Fare is not a statistically significant predictor here.
3. **C(sex)[T.male] (negative, p<0.001):** Being male significantly decreased survival odds.
4. **pclass (negative, p=0.077):** Lower passenger class is associated with lower survival odds, but it is not significant at the 0.05 level.

**P-values (P>|z|):**

- Values < 0.05 indicate statistical significance at the 5% level.
- Here, **sex** and **age** are significant; **fare** and **pclass** are not (at 0.05).



### Prediction from Statsmodels

Now that we've fitted our logistic regression model, let's generate predictions.

In `statsmodels`, the `.predict()` method returns **predicted probabilities** for the positive class (survived = 1). These are continuous values between 0 and 1, representing the model's estimated probability that each observation belongs to the positive class.

Unlike sklearn, statsmodels does not have a built-in method to directly return class labels (0/1)—we'll need to apply a threshold manually if we want binary predictions.

In [10]:
# Get predictions from statsmodels
train_probs_sm = statsmodels_logit.predict(train_df)
test_probs_sm = statsmodels_logit.predict(test_df)

print("Sample of predicted probabilities (statsmodels):")
test_probs_sm.head(10)

Sample of predicted probabilities (statsmodels):


565    0.094223
160    0.044466
553    0.100257
860    0.049784
241    0.606897
559    0.464106
387    0.755221
536    0.364969
698    0.341170
99     0.199713
dtype: float64

**Why this design?**

Statsmodels is more focused on:

* Statistical inference
* Coefficient interpretation
* Standard errors / p-values

It leaves classification decisions (thresholding, ROC analysis, etc.) to the user.

In [11]:
# Apply threshold to convert probabilities to class predictions
threshold = 0.5
train_pred_sm = (train_probs_sm >= threshold).astype(int)
test_pred_sm = (test_probs_sm >= threshold).astype(int)

# Calculate accuracy
train_accuracy_sm = accuracy_score(train_df['survived'], train_pred_sm)
test_accuracy_sm = accuracy_score(test_df['survived'], test_pred_sm)

print(f"Statsmodels Logistic Regression (threshold = {threshold}):")
print(f"  Training Accuracy:   {train_accuracy_sm:.4f} ({train_accuracy_sm:.2%})")
print(f"  Test Accuracy:       {test_accuracy_sm:.4f} ({test_accuracy_sm:.2%})")

# Show some example predictions
print(f"\nSample predictions (first 10 test observations):")
print(f"{'Index':<8} {'Probability':<12} {'Prediction':<12} {'Actual':<8}")
print("-" * 45)
for i in range(10):
    print(f"{i:<8} {test_probs_sm.iloc[i]:<12.4f} {test_pred_sm.iloc[i]:<12} {test_df['survived'].iloc[i]:<8}")

Statsmodels Logistic Regression (threshold = 0.5):
  Training Accuracy:   0.7992 (79.92%)
  Test Accuracy:       0.7821 (78.21%)

Sample predictions (first 10 test observations):
Index    Probability  Prediction   Actual  
---------------------------------------------
0        0.0942       0            0       
1        0.0445       0            0       
2        0.1003       0            1       
3        0.0498       0            0       
4        0.6069       1            1       
5        0.4641       0            1       
6        0.7552       1            1       
7        0.3650       0            0       
8        0.3412       0            0       
9        0.1997       0            0       


##  Implementation 2: Scikit-learn Integration

While `statsmodels` is excellent for **statistical inference** (p-values, confidence intervals), `sklearn` is optimized for **prediction and machine learning workflows**.

To use `sklearn.linear_model.LogisticRegression`, we have two approaches:

**Approach 1: Manual Dummy Variable Creation**

1. Manually create dummy variables using `pd.get_dummies()` (sklearn doesn't have formula notation)
2. Prepare feature matrix X and target vector y

**Approach 2: Pipeline with `ColumnTransformer`**

- Use sklearn's preprocessing pipeline to handle categorical encoding automatically
- More scalable for production workflows

We'll start with the manual approach first, then demonstrate the pipeline method at the end.

In [12]:
# Prepare data for sklearn using the same predictors as statsmodels
predictors = ['age', 'fare', 'sex', 'pclass']

X_train = pd.get_dummies(train_df[predictors], columns=['sex' ], drop_first=True)
X_test = pd.get_dummies(test_df[predictors], columns=['sex' ], drop_first=True)
X_test = X_test.reindex(columns=X_train.columns, fill_value=0)

y_train = train_df['survived']
y_test = test_df['survived']

print("Encoded training data:")
X_train.head()

print(f"\nColumn names: {list(X_train.columns)}")


Encoded training data:

Column names: ['age', 'fare', 'pclass', 'sex_male']


In [13]:
# Feature/target shapes
print(f"X_train shape: {X_train.shape}")
print(f"y_train shape: {y_train.shape}")
print(f"\nFeature names: {list(X_train.columns)}")


X_train shape: (712, 4)
y_train shape: (712,)

Feature names: ['age', 'fare', 'pclass', 'sex_male']


In [21]:
# Fit logistic regression using sklearn
sklearn_logit = LogisticRegression(random_state=42)
sklearn_logit.fit(X_train, y_train)

print("Sklearn Logistic Regression model fitted successfully!")
print(f"\nCoefficients:")
for feature, coef in zip(X_train.columns, sklearn_logit.coef_[0]):
    print(f"  {feature:20s}: {coef:8.4f}")
print(f"\nIntercept: {sklearn_logit.intercept_[0]:.4f}")

Sklearn Logistic Regression model fitted successfully!

Coefficients:
  age                 :  -0.0389
  fare                :   0.0010
  pclass              :  -1.2189
  sex_male            :  -2.4830

Intercept: 4.8699


###   Understanding `.predict()` vs `.predict_proba()`

Scikit-learn's logistic regression offers two prediction methods:
   
`.predict_proba()` gives you the raw probabilities, while `.predict()` gives you the final classification decision using default threshold of 0.5 (if prob ≥ 0.5, predict 1).


In [22]:
# Get probability predictions
test_probs_sklearn = sklearn_logit.predict_proba(X_test)

print("Shape of predict_proba output:", test_probs_sklearn.shape)
print("\nFirst 10 probability predictions:")
print("Index | P(Died) | P(Survived)")
print("-" * 35)
for i in range(10):
    print(f"{i:5d} | {test_probs_sklearn[i, 0]:.4f}  | {test_probs_sklearn[i, 1]:.4f}")

# Extract probabilities of survival (class 1)
test_probs_survival = test_probs_sklearn[:, 1]
print(f"\nProbabilities of survival (class 1): {test_probs_survival[:5]}")

Shape of predict_proba output: (179, 2)

First 10 probability predictions:
Index | P(Died) | P(Survived)
-----------------------------------
    0 | 0.8983  | 0.1017
    1 | 0.9510  | 0.0490
    2 | 0.8927  | 0.1073
    3 | 0.9454  | 0.0546
    4 | 0.4031  | 0.5969
    5 | 0.5424  | 0.4576
    6 | 0.2603  | 0.7397
    7 | 0.6355  | 0.3645
    8 | 0.6512  | 0.3488
    9 | 0.7937  | 0.2063

Probabilities of survival (class 1): [0.10166063 0.0490081  0.10730699 0.0546384  0.59687177]


In [23]:
# Get class predictions (using default 0.5 threshold)
test_pred_default = sklearn_logit.predict(X_test)

print("Shape of predict output:", test_pred_default.shape)
print("\nFirst 10 class predictions:")
print(test_pred_default[:10])

# Compare with probabilities
print("\nComparison of probabilities and predictions:")
print("Index | Probability | Prediction | Interpretation")
print("-" * 60)
for i in range(10):
    prob = test_probs_survival[i]
    pred = test_pred_default[i]
    interpretation = "Survived" if pred == 1 else "Died"
    print(f"{i:5d} | {prob:11.4f} | {pred:10d} | {interpretation}")



Shape of predict output: (179,)

First 10 class predictions:
[0 0 0 0 1 0 1 0 0 0]

Comparison of probabilities and predictions:
Index | Probability | Prediction | Interpretation
------------------------------------------------------------
    0 |      0.1017 |          0 | Died
    1 |      0.0490 |          0 | Died
    2 |      0.1073 |          0 | Died
    3 |      0.0546 |          0 | Died
    4 |      0.5969 |          1 | Survived
    5 |      0.4576 |          0 | Died
    6 |      0.7397 |          1 | Survived
    7 |      0.3645 |          0 | Died
    8 |      0.3488 |          0 | Died
    9 |      0.2063 |          0 | Died


**Observation:** Notice how `.predict()` converts probabilities to 0/1 using the 0.5 threshold:

- Probability ≥ 0.5 → Prediction = 1 (Survived)
- Probability < 0.5 → Prediction = 0 (Died)


### Turning off the default regularization

**Important difference between statsmodels and sklearn:**

By default, sklearn's `LogisticRegression` applies **L2 regularization** (ridge penalty) with `C=1.0`. This means:

- The model penalizes large coefficients to prevent overfitting
- Coefficients will be different from statsmodels (which uses no regularization by default)

**Understanding the `C` parameter:**

- `C` is the **inverse of regularization strength** (higher C = less regularization)
- Default: `C=1.0` applies moderate regularization
- To turn off regularization: Set `C=np.inf` (infinite C = no penalty)

- Note: We'll cover regularization and tuning C in detail in a later lectureLet's refit the model without regularization and compare the coefficients:

To match statsmodels' behavior and get coefficients that are directly comparable, we need to:

- Set `C=np.inf` to turn off regularization completely

In [24]:
# Fit logistic regression WITHOUT regularization (C=np.inf)
sklearn_logit_no_reg = LogisticRegression(C=np.inf, random_state=42)
sklearn_logit_no_reg.fit(X_train, y_train)

print("Sklearn Logistic Regression (NO regularization, C=np.inf):")
print(f"\nCoefficients:")
for feature, coef in zip(X_train.columns, sklearn_logit_no_reg.coef_[0]):
    print(f"  {feature:20s}: {coef:8.4f}")
print(f"\nIntercept: {sklearn_logit_no_reg.intercept_[0]:.4f}")

Sklearn Logistic Regression (NO regularization, C=np.inf):

Coefficients:
  age                 :  -0.0400
  fare                :   0.0007
  pclass              :  -1.2733
  sex_male            :  -2.6032

Intercept: 5.1031


In [25]:
print("\n" + "="*60)
print("COMPARISON: With vs Without Regularization")
print("="*60)
print(f"{'Feature':<20} {'With Reg (C=1.0)':<18} {'No Reg (C=np.inf)':<18}")
print("-"*60)
for i, feature in enumerate(X_train.columns):
    coef_with = sklearn_logit.coef_[0][i]
    coef_without = sklearn_logit_no_reg.coef_[0][i]
    print(f"{feature:<20} {coef_with:17.4f} {coef_without:17.4f}")
    
print(f"\n{'Intercept':<20} {sklearn_logit.intercept_[0]:17.4f} {sklearn_logit_no_reg.intercept_[0]:17.4f}")


COMPARISON: With vs Without Regularization
Feature              With Reg (C=1.0)   No Reg (C=np.inf) 
------------------------------------------------------------
age                            -0.0389           -0.0400
fare                            0.0010            0.0007
pclass                         -1.2189           -1.2733
sex_male                       -2.4830           -2.6032

Intercept                       4.8699            5.1031


**Key Observations:**

1. **With regularization (default `C=1.0`)**: Coefficients are slightly shrunk toward zero
2. **Without regularization (`C=np.inf`)**: Coefficients match statsmodels more closely

**When to use which approach:**

- **Use regularization (smaller C values)**: 
  - When you have many features or potential overfitting
  - For production models where generalization is key
  - Default sklearn behavior (`C=1.0`) is good for most machine learning workflows

- **Turn off regularization (`C=np.inf`)**:
  - When you want to match statistical software like statsmodels or R
  - For small datasets with few predictors
  - When interpretability and hypothesis testing are priorities
  - When comparing results across different tools

**Note:** We'll explore how to choose optimal C values using cross-validation in the regularization lecture.

In [27]:
# the training and test accuracy for the model with and without regularization
train_pred_no_reg = sklearn_logit_no_reg.predict(X_train)
test_pred_no_reg = sklearn_logit_no_reg.predict(X_test)
train_accuracy_no_reg = accuracy_score(y_train, train_pred_no_reg)
test_accuracy_no_reg = accuracy_score(y_test, test_pred_no_reg)

train_pred_with_reg = sklearn_logit.predict(X_train)
test_pred_with_reg = sklearn_logit.predict(X_test)
train_accuracy_with_reg = accuracy_score(y_train, train_pred_with_reg)
test_accuracy_with_reg = accuracy_score(y_test, test_pred_with_reg)

print(f"\nSklearn Logistic Regression WITHOUT regularization (C=np.inf):")
print(f"  Training Accuracy:   {train_accuracy_no_reg:.4f} ({train_accuracy_no_reg:.2%})")
print(f"  Test Accuracy:       {test_accuracy_no_reg:.4f} ({test_accuracy_no_reg:.2%})")

print(f"\nSklearn Logistic Regression WITH regularization:")
print(f"  Training Accuracy:   {train_accuracy_with_reg:.4f} ({train_accuracy_with_reg:.2%})")
print(f"  Test Accuracy:       {test_accuracy_with_reg:.4f} ({test_accuracy_with_reg:.2%})")


Sklearn Logistic Regression WITHOUT regularization (C=np.inf):
  Training Accuracy:   0.7992 (79.92%)
  Test Accuracy:       0.7821 (78.21%)

Sklearn Logistic Regression WITH regularization:
  Training Accuracy:   0.8006 (80.06%)
  Test Accuracy:       0.7821 (78.21%)


### Pipeline Implementation

**Why use Pipelines?**

So far, we've manually created dummy variables using `pd.get_dummies()`. While this works, sklearn's **Pipeline** approach offers several advantages:

1. **Automated preprocessing**: Handles categorical encoding automatically
2. **Prevents data leakage**: Ensures transformations are fit only on training data
3. **Cleaner code**: Combines preprocessing and modeling in one object
4. **Production-ready**: Easier to deploy and maintain
5. **Reproducibility**: All steps are encapsulated in one pipeline

**Key components:**
- `ColumnTransformer`: Applies different transformations to different columns
- `OneHotEncoder`: Converts categorical variables to dummy variables
- `Pipeline`: Chains preprocessing and model fitting together

Let's implement the same logistic regression model using a pipeline (without regularization to match statsmodels):

In [29]:
# Import required components for pipeline
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder

# Define which columns are categorical and which are numeric
categorical_features = ['sex']
numeric_features = ['age', 'fare', 'pclass']

# Create the column transformer
# OneHotEncoder with drop='first' mimics the drop_first=True in pd.get_dummies
preprocessor = ColumnTransformer(
    transformers=[
        ('num', 'passthrough', numeric_features),  # Keep numeric features as-is
        ('cat', OneHotEncoder(drop='first'), categorical_features)  # Encode categorical
    ])

# Create the pipeline: preprocessing + logistic regression
pipeline_model = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', LogisticRegression(C=np.inf, random_state=42))
])

# Fit the pipeline on training data
# Note: We pass the original dataframe with categorical variables
X_train_raw = train_df[['age', 'fare', 'sex', 'pclass']]
X_test_raw = test_df[['age', 'fare', 'sex', 'pclass']]

pipeline_model.fit(X_train_raw, y_train)

print("Pipeline model fitted successfully!")
print("\nPipeline structure:")
print(pipeline_model)

Pipeline model fitted successfully!

Pipeline structure:
Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('num', 'passthrough',
                                                  ['age', 'fare', 'pclass']),
                                                 ('cat',
                                                  OneHotEncoder(drop='first'),
                                                  ['sex'])])),
                ('classifier', LogisticRegression(C=inf, random_state=42))])


In [30]:
# Extract the trained logistic regression model from the pipeline
pipeline_logit = pipeline_model.named_steps['classifier']

# Get feature names after transformation
feature_names = numeric_features + ['sex_male']  # OneHotEncoder creates sex_male (drops sex_female)

print("Pipeline Logistic Regression Coefficients (C=np.inf):")
print(f"\nCoefficients:")
for feature, coef in zip(feature_names, pipeline_logit.coef_[0]):
    print(f"  {feature:20s}: {coef:8.4f}")
print(f"\nIntercept: {pipeline_logit.intercept_[0]:.4f}")

# Make predictions
pipeline_train_pred = pipeline_model.predict(X_train_raw)
pipeline_test_pred = pipeline_model.predict(X_test_raw)

# Calculate accuracy
pipeline_train_accuracy = accuracy_score(y_train, pipeline_train_pred)
pipeline_test_accuracy = accuracy_score(y_test, pipeline_test_pred)

print(f"\nPipeline Model Performance:")
print(f"  Training Accuracy:   {pipeline_train_accuracy:.4f} ({pipeline_train_accuracy:.2%})")
print(f"  Test Accuracy:       {pipeline_test_accuracy:.4f} ({pipeline_test_accuracy:.2%})")

Pipeline Logistic Regression Coefficients (C=np.inf):

Coefficients:
  age                 :  -0.0400
  fare                :   0.0007
  pclass              :  -1.2733
  sex_male            :  -2.6032

Intercept: 5.1031

Pipeline Model Performance:
  Training Accuracy:   0.7992 (79.92%)
  Test Accuracy:       0.7821 (78.21%)


In [31]:
# Verify that pipeline and manual approach give identical results
print("="*70)
print("COMPARISON: Pipeline vs Manual Approach (both with C=np.inf)")
print("="*70)
print(f"\n{'Feature':<20} {'Manual Approach':<18} {'Pipeline Approach':<18} {'Match?':<8}")
print("-"*70)

# Compare intercept
manual_intercept = sklearn_logit_no_reg.intercept_[0]
pipeline_intercept = pipeline_logit.intercept_[0]
match = "✓" if np.isclose(manual_intercept, pipeline_intercept) else "✗"
print(f"{'Intercept':<20} {manual_intercept:17.6f} {pipeline_intercept:17.6f} {match:<8}")

# Compare coefficients
for i, feature in enumerate(feature_names):
    manual_coef = sklearn_logit_no_reg.coef_[0][i]
    pipeline_coef = pipeline_logit.coef_[0][i]
    match = "✓" if np.isclose(manual_coef, pipeline_coef) else "✗"
    print(f"{feature:<20} {manual_coef:17.6f} {pipeline_coef:17.6f} {match:<8}")

print("\n" + "="*70)
print("RESULT: Both approaches produce IDENTICAL coefficients!")
print("="*70)

COMPARISON: Pipeline vs Manual Approach (both with C=np.inf)

Feature              Manual Approach    Pipeline Approach  Match?  
----------------------------------------------------------------------
Intercept                     5.103138          5.103138 ✓       
age                          -0.039957         -0.039957 ✓       
fare                          0.000660          0.000660 ✓       
pclass                       -1.273318         -1.273318 ✓       
sex_male                     -2.603232         -2.603232 ✓       

RESULT: Both approaches produce IDENTICAL coefficients!


**Key Advantages of Pipeline Approach:**

1. **Identical Results**: Pipeline produces the exact same coefficients and predictions as the manual approach
2. **Cleaner Code**: All preprocessing and modeling in one object
3. **No Data Leakage**: The pipeline ensures that:
   - OneHotEncoder is fit only on training data
   - Categories seen only in test data are handled correctly
4. **Production Ready**: Can be easily saved and loaded for deployment
5. **Works with Cross-Validation**: Seamlessly integrates with GridSearchCV and other sklearn tools

**When to use each approach:**

- **Manual `pd.get_dummies()`**: 
  - Quick exploratory analysis
  - When you need fine control over encoding
  - Simple, one-off models

- **Pipeline with ColumnTransformer**:
  - Production environments
  - When using cross-validation or hyperparameter tuning
  - Multiple preprocessing steps
  - When you need to ensure reproducibility and prevent data leakage

**In the next 2 weeks, you will learn**

- Use pipelines with cross-validation to tune hyperparameters (like `C`)
- Extend pipelines with additional preprocessing (scaling, imputation, feature engineering)
- Save trained pipelines for deployment using `joblib` or `pickle`

## Statsmodels vs Sklearn Result Comparison

**Why might coefficients differ?**

Unlike linear regression which has a closed-form solution (matrix operations), logistic regression requires **iterative optimization algorithms** to find the best coefficients. Both statsmodels and sklearn use iterative methods, but:

- **Statsmodels** uses: Newton-Raphson/IRLS (Iteratively Reweighted Least Squares)
- **Sklearn** uses: LBFGS, liblinear, SAG, or SAGA (depending on settings)

Even with `C=np.inf` (no regularization), we might see **slight differences** due to:
1. Different optimization algorithms
2. Different convergence criteria
3. Different tolerance thresholds
4. Numerical precision differences

However, when both converge properly, the coefficients should be **very close** (differences typically in the 3rd-4th decimal place or smaller).

Let's compare the coefficients from statsmodels and sklearn (without regularization):

In [28]:
# Extract coefficients from statsmodels
# Note: statsmodels coefficients are in a different order due to formula notation
statsmodels_coefs = statsmodels_logit.params

print("="*70)
print("COEFFICIENT COMPARISON: Statsmodels vs Sklearn (C=np.inf)")
print("="*70)
print(f"\n{'Variable':<20} {'Statsmodels':<18} {'Sklearn (C=np.inf)':<18} {'Difference':<12}")
print("-"*70)

# Compare intercept
sklearn_intercept = sklearn_logit_no_reg.intercept_[0]
statsmodels_intercept = statsmodels_coefs['Intercept']
diff_intercept = abs(sklearn_intercept - statsmodels_intercept)
print(f"{'Intercept':<20} {statsmodels_intercept:17.6f} {sklearn_intercept:17.6f} {diff_intercept:11.6f}")

# Compare age coefficient
sklearn_age = sklearn_logit_no_reg.coef_[0][list(X_train.columns).index('age')]
statsmodels_age = statsmodels_coefs['age']
diff_age = abs(sklearn_age - statsmodels_age)
print(f"{'age':<20} {statsmodels_age:17.6f} {sklearn_age:17.6f} {diff_age:11.6f}")

# Compare fare coefficient
sklearn_fare = sklearn_logit_no_reg.coef_[0][list(X_train.columns).index('fare')]
statsmodels_fare = statsmodels_coefs['fare']
diff_fare = abs(sklearn_fare - statsmodels_fare)
print(f"{'fare':<20} {statsmodels_fare:17.6f} {sklearn_fare:17.6f} {diff_fare:11.6f}")

# Compare sex coefficient (male)
sklearn_sex_male = sklearn_logit_no_reg.coef_[0][list(X_train.columns).index('sex_male')]
statsmodels_sex_male = statsmodels_coefs['C(sex)[T.male]']
diff_sex = abs(sklearn_sex_male - statsmodels_sex_male)
print(f"{'sex_male':<20} {statsmodels_sex_male:17.6f} {sklearn_sex_male:17.6f} {diff_sex:11.6f}")

# Compare pclass coefficient
sklearn_pclass = sklearn_logit_no_reg.coef_[0][list(X_train.columns).index('pclass')]
statsmodels_pclass = statsmodels_coefs['pclass']
diff_pclass = abs(sklearn_pclass - statsmodels_pclass)
print(f"{'pclass':<20} {statsmodels_pclass:17.6f} {sklearn_pclass:17.6f} {diff_pclass:11.6f}")

print("\n" + "="*70)
print("INTERPRETATION:")
print("="*70)
print("✓ Coefficients are very close (differences < 0.001 typically)")
print("✓ Both models have converged to essentially the same solution")
print("✓ Minor differences are due to numerical optimization details")
print("✓ For practical purposes, both implementations give equivalent results")
print("="*70)

COEFFICIENT COMPARISON: Statsmodels vs Sklearn (C=np.inf)

Variable             Statsmodels        Sklearn (C=np.inf) Difference  
----------------------------------------------------------------------
Intercept                     5.103261          5.103138    0.000123
age                          -0.039956         -0.039957    0.000001
fare                          0.000659          0.000660    0.000001
sex_male                     -2.603225         -2.603232    0.000007
pclass                       -1.273379         -1.273318    0.000062

INTERPRETATION:
✓ Coefficients are very close (differences < 0.001 typically)
✓ Both models have converged to essentially the same solution
✓ Minor differences are due to numerical optimization details
✓ For practical purposes, both implementations give equivalent results


**When do these differences matter?**

In most practical applications, these tiny differences are **negligible**:

- Predictions will be virtually identical
- Model performance metrics will be the same
- Interpretation of coefficients remains consistent

**Key takeaway:**

- When comparing results across tools, use `C=np.inf` in sklearn to match statsmodels
- Both implementations are correct and reliable
- Choose the tool based on your workflow needs (statsmodels for inference, sklearn for ML pipelines)